# 03 — GRU Diagnostics

Analyze GRU routing quality: weight distribution, prediction accuracy,
confusion patterns, and softmax behavior.

**Requires:** `vault-exec init` (KV with GRU weights + traces)

In [1]:
import { openVaultStore } from "../src/db/index.ts";
import { GRUInference } from "../src/gru/inference.ts";
import { deserializeWeights } from "../src/gru/weights.ts";

const VAULT_PATH = "../demo-vault";
const DB_PATH = `${VAULT_PATH}/.vault-exec/vault.kv`;

const db = await openVaultStore(DB_PATH);

// Load GRU weights
const weightsRow = await db.getLatestWeights();
if (!weightsRow) throw new Error("No GRU weights — run init first");

const { weights, vocab, config } = await deserializeWeights(weightsRow.blob);
const gru = new GRUInference(weights, vocab, config);

console.log(`GRU loaded: ${vocab.nodes.length} vocab nodes`);
console.log(`Config: inputDim=${config.inputDim}, hiddenDim=${config.hiddenDim}`);


GRU loaded: 124 vocab nodes


Config: inputDim=1024, hiddenDim=32


In [2]:
// Load all traces from DB
const traces = await db.getAllTraces();
const synthetic = traces.filter(t => t.synthetic);
const real = traces.filter(t => !t.synthetic);

console.log(`Total traces: ${traces.length}`);
console.log(`  Synthetic: ${synthetic.length}`);
console.log(`  Real:      ${real.length}`);

// Target distribution
const targetCounts = new Map<string, number>();
for (const t of traces) {
  targetCounts.set(t.targetNote, (targetCounts.get(t.targetNote) ?? 0) + 1);
}

console.log('\nTarget distribution:');
[...targetCounts.entries()].sort((a, b) => b[1] - a[1]).forEach(([name, count]) => {
  console.log(`  ${name}: ${count}x`);
});

Total traces: 137


  Synthetic: 98


  Real:      39



Target distribution:


  Dispatch Dry Run: 4x


...

  REV - Renewal Meeting Payload: 1x


  REV - Renewal Outreach Drafts: 1x


  REV - Task Close Payload: 1x


  REV - Task Reassign Payload: 1x


  REV - Task Upsert Payload: 1x


In [3]:
// Target frequency bar chart
const targetData = [...targetCounts.entries()].map(([name, count]) => ({
  note: name,
  count,
  type: synthetic.some(t => t.targetNote === name) ? 'synthetic' : 'real',
})).sort((a, b) => b.count - a.count);

const targetSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  "title": "Training Target Distribution",
  "width": 500,
  "height": 250,
  "data": { "values": targetData },
  "mark": "bar",
  "encoding": {
    "x": { "field": "note", "type": "nominal", "sort": "-y", "title": "Target Note" },
    "y": { "field": "count", "type": "quantitative", "title": "Trace Count" },
    "color": { "field": "type", "type": "nominal" },
    "tooltip": [{ "field": "note" }, { "field": "count" }]
  }
};

await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": targetSpec }, { raw: true });

In [4]:
// GRU prediction test: for each note's embedding, what does the GRU predict?
const allNotes = await db.getAllNotes();
const predictions: Array<{note: string, predicted: string, score: number, correct: boolean}> = [];

for (const note of allNotes) {
  const emb = note.gnnEmbedding ?? note.embedding;
  if (!emb) continue;
  
  const prediction = gru.predictNext(emb, []);
  predictions.push({
    note: note.name,
    predicted: prediction.name,
    score: parseFloat(prediction.score.toFixed(4)),
    correct: prediction.name === note.name,
  });
}

console.log('\n=== GRU Self-Prediction (embed → predict) ===');
console.table(predictions);

const selfAccuracy = predictions.filter(p => p.correct).length / predictions.length;
console.log(`\nSelf-prediction accuracy: ${(selfAccuracy * 100).toFixed(1)}%`);


=== GRU Self-Prediction (embed → predict) ===


┌───────┬──────────────────────────────────────┬────────────────────────────┬────────┬─────────┐
│ (idx) │ note                                 │ predicted                  │ score  │ correct │
├───────┼──────────────────────────────────────┼────────────────────────────┼────────┼─────────┤
│     0 │ "Account Contract Links"             │ "Upsell Candidate Queue"   │ 0.2566 │ false   │
│     1 │ "Account Health Inputs"              │ "Upsell Candidate Queue"   │ 0.192  │ false   │
│     2 │ "Account Ticket Links"               │ "Upsell Candidate Queue"   │ 0.2461 │ false   │
│     3 │ "Account Tier Weight"                │ "Upsell Candidate Queue"   │ 0.2115 │ false   │
│     4 │ "Accounts"                           │ "Upsell Candidate Queue"   │ 0.2149 │ false   │
│     5 │ "Action Dispatch Queue"              │ "Upsell Candidate Queue"   │ 0.2234 │ false   │
│     6 │ "Action Eligibility"                 │ "Upsell Candidate Queue"   │ 0.1743 │ false   │
│     7 │ "Churn Pressure Scor


Self-prediction accuracy: 0.8%


In [5]:
// Score distribution across vocab for each prediction
// Shows if the GRU is confident or spread across many notes
const scoreDistributions: Array<{note: string, targetNote: string, score: number}> = [];

for (const note of allNotes.slice(0, 5)) { // first 5 notes
  const emb = note.gnnEmbedding ?? note.embedding;
  if (!emb) continue;
  
  // Get top-K predictions
  for (const vn of vocab.nodes) {
    const pred = gru.predictNext(emb, []);
    scoreDistributions.push({
      note: note.name,
      targetNote: vn.name,
      score: pred.score,
    });
  }
}

console.log(`Score distribution computed for ${Math.min(5, allNotes.length)} notes`);

Score distribution computed for 5 notes


In [6]:
// Confusion matrix: for real traces with intent, what did GRU predict vs actual target?
const confusionData: Array<{actual: string, predicted: string, count: number}> = [];
const confMap = new Map<string, number>();

for (const trace of real) {
  if (!trace.intentEmbedding) continue;
  const pred = gru.predictNext(trace.intentEmbedding, []);
  const key = `${trace.targetNote}||${pred.name}`;
  confMap.set(key, (confMap.get(key) ?? 0) + 1);
}

for (const [key, count] of confMap) {
  const [actual, predicted] = key.split('||');
  confusionData.push({ actual, predicted, count });
}

if (confusionData.length > 0) {
  const confSpec = {
    "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
    "title": "GRU Routing Confusion (Real Traces)",
    "width": 400,
    "height": 400,
    "data": { "values": confusionData },
    "mark": "rect",
    "encoding": {
      "x": { "field": "predicted", "type": "nominal", "title": "GRU Predicted" },
      "y": { "field": "actual", "type": "nominal", "title": "Actual Target" },
      "color": { "field": "count", "type": "quantitative", "scale": { "scheme": "blues" } },
      "tooltip": [{ "field": "actual" }, { "field": "predicted" }, { "field": "count" }]
    }
  };
  await Deno.jupyter.display({ "application/vnd.vegalite.v5+json": confSpec }, { raw: true });
} else {
  console.log('No real traces with intent embeddings yet — run some --intent queries first.');
}

In [7]:
// Weight analysis: GRU hidden state and output projection norms
function matrixNorm(arr: number[][]): number {
  let sum = 0;
  for (const row of arr) for (const v of row) sum += v * v;
  return Math.sqrt(sum);
}

console.log('\n=== GRU Weight Statistics ===');
for (const [name, w] of Object.entries(weights)) {
  if (Array.isArray(w) && Array.isArray(w[0])) {
    const m = w as number[][];
    console.log(`${name}: [${m.length} × ${m[0].length}], Frobenius norm=${matrixNorm(m).toFixed(4)}`);
  } else if (Array.isArray(w)) {
    const v = w as number[];
    console.log(`${name}: [${v.length}], L2 norm=${Math.sqrt(v.reduce((s, x) => s + x * x, 0)).toFixed(4)}`);
  }
}


=== GRU Weight Statistics ===


W_input: [64 × 1024], Frobenius norm=6.4563


b_input: [64], L2 norm=0.0154


W_z: [32 × 64], Frobenius norm=3.7271


b_z: [32], L2 norm=0.0402


...

b_intent: [64], L2 norm=0.0183


W_fusion: [32 × 96], Frobenius norm=3.9766


b_fusion: [32], L2 norm=0.0123


W_output: [1024 × 32], Frobenius norm=4.6179


b_output: [1024], L2 norm=0.0643


In [8]:
db.close();
console.log('Done');

Done
